In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import shutil
import subprocess
import time
from pathlib import Path

In [ ]:
FOLDER_ID = "1xniAy8v6QDP8FxTkUFQSWH-3rftx7n39"
FOLDER_NAME = "AIMOProofPilot Container"
ZIP_NAME = "AIMOProofPilot_Container.zip"
RCLONE_REMOTE = "gdrive"
DUMMY_TEST = True
DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT / "MyDrive"
PACKAGE_DIR = DRIVE_ROOT / FOLDER_NAME
LOCAL_SHA256_PATH = Path("/content") / f"{ZIP_NAME}.sha256"
RCLONE_REMOTE_ZIP_PATH = f"{RCLONE_REMOTE}:{ZIP_NAME}"
RCLONE_REMOTE_SHA256_PATH = f"{RCLONE_REMOTE}:{ZIP_NAME}.sha256"
EXPECTED_SIF_NAME = "aimo-proof_train_20260611.sif"
EXPECTED_DEF_NAME = "aimo-proof_train_20260611.def"
DATASET_NAME = "aimo_proof_train.parquet"

In [ ]:
def assert_drive_path(path: Path, name: str) -> None:

    resolved_path = path.resolve()
    resolved_mount = DRIVE_MOUNT.resolve()

    try:
        resolved_path.relative_to(resolved_mount)
    except ValueError as error:
        raise ValueError(f"{name} must be under {resolved_mount}, got {resolved_path}") from error


assert_drive_path(PACKAGE_DIR, "PACKAGE_DIR")

if not PACKAGE_DIR.exists():
    raise FileNotFoundError(
        f"Package folder was not found at {PACKAGE_DIR}. "
        "If this is a shared Drive folder, add it as a shortcut to My Drive "
        f"from https://drive.google.com/drive/folders/{FOLDER_ID}, "
        "or edit PACKAGE_DIR to the mounted path Colab sees."
    )

if not PACKAGE_DIR.is_dir():
    raise NotADirectoryError(f"Package path is not a directory: {PACKAGE_DIR}")

print(f"Package directory: {PACKAGE_DIR}")
print(f"Zip upload target: {RCLONE_REMOTE_ZIP_PATH}")
print(f"Checksum upload target: {RCLONE_REMOTE_SHA256_PATH}")

In [ ]:
required_paths = [
    Path("README.md"),
    Path("upload.py"),
    Path("requirements_container.txt"),
    Path(EXPECTED_SIF_NAME),
    Path(EXPECTED_DEF_NAME),
    Path("dataset") / DATASET_NAME,
]

if DUMMY_TEST:
    model_paths = [Path("models") / "dummy"]
else:
    model_paths = [
        Path("models") / "contestant",
        Path("models") / "judge",
    ]

missing_paths = [
    str(path)
    for path in required_paths + model_paths
    if not (PACKAGE_DIR / path).exists()
]

if missing_paths:
    raise FileNotFoundError("Missing required package paths: " + json.dumps(missing_paths, indent=2))

model_safetensor_counts = {
    path.as_posix(): len(sorted((PACKAGE_DIR / path).glob("*.safetensors")))
    for path in model_paths
}

print(f"Dummy test mode: {DUMMY_TEST}")
print("Model folders: " + json.dumps([path.as_posix() for path in model_paths]))
print("Model safetensor counts: " + json.dumps(model_safetensor_counts, indent=2))

In [ ]:
file_entries = []
total_bytes = 0

for path in sorted(PACKAGE_DIR.rglob("*")):
    if not path.is_file():
        continue

    if any(part in {".hf_cache", ".tmp", ".zip_tmp", "__pycache__"} for part in path.relative_to(PACKAGE_DIR).parts):
        continue

    if path.suffix == ".zip":
        continue

    size_bytes = path.stat().st_size
    total_bytes += size_bytes
    file_entries.append({
        "path": path.relative_to(PACKAGE_DIR).as_posix(),
        "size_bytes": size_bytes,
    })

manifest = {
    "created_at_unix": time.time(),
    "folder_id": FOLDER_ID,
    "package_dir": str(PACKAGE_DIR),
    "zip_name": ZIP_NAME,
    "rclone_remote_zip_path": RCLONE_REMOTE_ZIP_PATH,
    "rclone_remote_sha256_path": RCLONE_REMOTE_SHA256_PATH,
    "dummy_test": DUMMY_TEST,
    "local_disk_policy": "The ZIP stream is uploaded through rclone rcat, so no local archive is created.",
    "file_count": len(file_entries),
    "total_bytes": total_bytes,
    "model_safetensor_counts": model_safetensor_counts,
    "files": file_entries,
}

manifest_path = PACKAGE_DIR / "package_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

print(f"Package files: {len(file_entries)}")
print(f"Package bytes: {total_bytes}")
print(f"Manifest written: {manifest_path}")

In [ ]:
if shutil.which("rclone") is None:
    subprocess.run(
        "curl https://rclone.org/install.sh | sudo bash",
        shell=True,
        text=True,
        check=True,
    )

subprocess.run(["rclone", "version"], text=True, check=True)

In [ ]:
!rclone config

In [ ]:
completed_process = subprocess.run(
    ["rclone", "listremotes"],
    capture_output=True,
    text=True,
    check=False,
)

if completed_process.returncode != 0:
    raise RuntimeError(completed_process.stderr)

remote_names = set(completed_process.stdout.splitlines())

if f"{RCLONE_REMOTE}:" not in remote_names:
    raise RuntimeError(
        "Run the previous `!rclone config` cell, create a Google Drive remote named `gdrive`, "
        "choose `n` for auto config in Colab, authorize in the browser, "
        "paste the token back, finish the prompts, then rerun this cell."
    )

completed_process = subprocess.run(
    ["rclone", "lsd", f"{RCLONE_REMOTE}:"],
    text=True,
    check=False,
)

if completed_process.returncode != 0:
    raise RuntimeError(f"rclone could not list {RCLONE_REMOTE}:")

In [ ]:
%%bash
set -euo pipefail
DRIVE_ROOT="/content/drive/MyDrive"
FOLDER_NAME="AIMOProofPilot Container"
ZIP_NAME="AIMOProofPilot_Container.zip"
HASH_PATH="/content/${ZIP_NAME}.sha256"
cd "$DRIVE_ROOT"
zip -q -0 -r - "$FOLDER_NAME" \
    -x "$FOLDER_NAME/.hf_cache/*" \
    -x "$FOLDER_NAME/.tmp/*" \
    -x "$FOLDER_NAME/.zip_tmp/*" \
    -x "$FOLDER_NAME/__pycache__/*" \
    -x "$FOLDER_NAME/*.zip" \
| tee >(sha256sum | sed "s|-|  ${ZIP_NAME}|" > "$HASH_PATH") \
| rclone rcat --progress --drive-chunk-size 256M "gdrive:$ZIP_NAME"
rclone copyto "$HASH_PATH" "gdrive:${ZIP_NAME}.sha256"

In [ ]:
completed_process = subprocess.run(
    ["rclone", "lsf", f"{RCLONE_REMOTE}:", "--files-only"],
    capture_output=True,
    text=True,
    check=False,
)

if completed_process.returncode != 0:
    raise RuntimeError(completed_process.stderr)

remote_file_names = set(completed_process.stdout.splitlines())

for remote_file_name in [ZIP_NAME, f"{ZIP_NAME}.sha256"]:
    if remote_file_name not in remote_file_names:
        raise FileNotFoundError(f"Remote file was not found: {remote_file_name}")

    print(f"Verified remote file: {remote_file_name}")

completed_process = subprocess.run(
    ["rclone", "cat", RCLONE_REMOTE_SHA256_PATH],
    capture_output=True,
    text=True,
    check=False,
)

if completed_process.returncode != 0:
    raise RuntimeError(completed_process.stderr)

checksum_text = completed_process.stdout.strip()

if not checksum_text.endswith(f"  {ZIP_NAME}"):
    raise RuntimeError(f"Unexpected checksum file contents: {checksum_text}")

print(checksum_text)